## Cosmic Web - CppJIT - CUDA Kernel

[CppJIT](https://github.com/compiler-research/cppjit) compiles C++ and CUDA source inside the Python session, through [clang-repl](https://clang.llvm.org/docs/ClangRepl.html) and [CppInterOp](https://github.com/compiler-research/CppInterOp). There is no binding to write: `cppjit.gbl.<name>` is the compiled function or kernel, and CuPy's arrays and streams go straight into a launch. This is the *kernel* model. In this notebook we call CUB's histogram from C++ first, then write the kernel ourselves and measure what it costs.

### Table of Contents

1. [Setup](#1-setup)
2. [CUB from C++](#2-cub-from-c)
3. [Kernel](#3-kernel)
4. [Launch](#4-launch)
5. [Trace and bytes](#5-trace-and-bytes)
6. [Limitations and recap](#6-limitations-and-recap)

---

### 1. Setup

`import cppjit` starts a CUDA-enabled clang-repl in this process.

---

**Quick Docs**

- `cppjit.cppdef(src)`: parse C++ or CUDA source into the session; its functions and kernels appear as `cppjit.gbl.<name>`.
- `cppjit.CUDA_ENABLED`: `True` when the interpreter was created with CUDA support.

In [ ]:
import os
import time

import numpy as np
import cupy as cp

import cosmic_core as cw            # sets the interpreter's optimisation level before cppjit starts
import cppjit

assert cppjit.CUDA_ENABLED, 'this notebook needs the CUDA-enabled tutorial image'

N, K = cw.setup('cppjit_cub', f"interpreter arguments {os.environ['CPPINTEROP_EXTRA_INTERPRETER_ARGS']}")

### 2. CUB from C++

The histogram the previous notebook called from Python is `DeviceHistogram` from [CUB](https://github.com/NVIDIA/cccl/tree/main/cub). `cosmic_cub_histogram.cu` wraps it in one host function that takes the device pointers and the stream as integers. Let's compile it into the session, count the eight cell arrays of the CuPy reference with it in place of `cp.bincount`, and make sure it works.

---

**Quick Docs**

- `cppjit.gbl.cub_histogram(samples_ptr, hist_ptr, n, k_bins, stream)`: CUB's `HistogramEven` over `n` int32 samples into `k_bins` counters.
- `a.data.ptr`, `stream.ptr`: a CuPy array's device address and a stream's handle, as integers.

In [ ]:
t0 = time.perf_counter()
cppjit.cppdef(cw.cub_histogram_source())
cppdef_cub_s = time.perf_counter() - t0
print(f'cppdef: {cppdef_cub_s * 1e3:.0f} ms to parse the CUB headers and compile the function')

SIGNS = (+1, -1, -1, -1, +1, +1, +1, -1)
CUB_HISTS = [cp.zeros(K, dtype=cp.int32) for _ in SIGNS]


def ecc_hist_cub(q):
    '''Signed histogram of births (host int64, length K): the CuPy stencil, CUB's histogram from C++ per cell array.'''
    v, (ex, ey, ez), (fxy, fxz, fyz), c = cw.ecc_cells_cupy(q)
    stream = cp.cuda.get_current_stream()
    for a, h in zip((v, ex, ey, ez, fxy, fxz, fyz, c), CUB_HISTS):
        cppjit.gbl.cub_histogram(a.data.ptr, h.data.ptr, a.size, K, stream.ptr)
    stream.synchronize()
    return sum(sign * cp.asnumpy(h).astype(np.int64) for sign, h in zip(SIGNS, CUB_HISTS))


def run_cub(n):
    return ecc_hist_cub(cw.field(n))


_ = cw.measure('cppjit_cub', run_cub, N, compile_s=cppdef_cub_s, cold_note='cppdef and first call')

### 3. Kernel

The kernel in `cosmic_ecc.cu` gives thread $t$ the voxel $(i, j, k)$ with $k$ fastest, so a *warp* (the 32 threads that execute together) reads consecutive addresses.

- Each thread loads its periodic $2\times2\times2$ window and takes the eight births with `min`.
- It adds $\pm 1$ at each birth bin into a `__shared__` histogram, the on-chip memory a block's threads share.
- After `__syncthreads()`, a barrier across the block, it adds its non-zero counters into the global histogram with `atomicAdd`.

Why shared memory? Each voxel would otherwise issue eight atomics into global memory. A block now issues at most $K$.

The TODO writes the body between the two markers. The zeroing, the periodic indices, the barrier and the flush are written.

---

**Quick Docs**

- `__global__ void kernel(...)`: a [CUDA C++](https://docs.nvidia.com/cuda/cuda-c-programming-guide/) kernel, run by a grid of blocks of threads; `blockIdx`, `blockDim` and `threadIdx` give each thread its index.
- `atomicAdd(int*, int)`, `atomicSub(int*, int)`: add or subtract at an address without a race, in shared or global memory.
- `cw.ecc_cuda_source(body)`: `cosmic_ecc.cu` with `body` spliced between its `BEGIN BODY` and `END BODY` markers.

In [ ]:
KERNEL_BODY = r'''
        // TODO: the eight births and the eight shared-memory atomics. The corners are q[i * nn + j * n + k] and its
        // neighbours at i1, j1, k1 (v000 ... v111); a cell is born at the minimum over its corners.
        // atomicAdd(&sh[b], 1) at v000, fxy, fxz, fyz; atomicSub(&sh[b], 1) at ex, ey, ez, c.
'''

t0 = time.perf_counter()
cppjit.cppdef(cw.ecc_cuda_source(body=KERNEL_BODY))
cppdef_s = time.perf_counter() - t0
print(f'cppdef: {cppdef_s * 1e3:.0f} ms to parse the unit and compile the device code to PTX')

### 4. Launch

The grid gives one thread per voxel. The shared histogram is static, so the dynamic shared-memory slot is zero. The stream slot takes CuPy's current stream. A CuPy array is accepted where the kernel takes a pointer.

The TODO writes the launch. The histogram buffer, the grid and the runner are written.

---

**Quick Docs**

- `cppjit.gbl.<kernel>[grid, block, shared_bytes, stream](*args)`: the launch; `grid` and `block` are ints or 3-tuples, `stream` an int handle or any `__cuda_stream__` object.
- `cp.zeros(K, dtype=cp.int32)`: the histogram buffer, one memset.

In [ ]:
BLOCK = 256


def ecc_hist_cppjit(q, block=BLOCK):
    '''Signed histogram of births (int32, length K) on the device: one launch on the current stream.'''
    n = q.shape[0]
    hist = cp.zeros(K, dtype=cp.int32)
    stream = cp.cuda.get_current_stream()
    grid = (n ** 3 + block - 1) // block
    # TODO: launch cppjit.gbl.ecc_hist_kernel on grid blocks of block threads, no dynamic shared memory, on stream, with q, hist and n.
    ...
    return hist


def run_cppjit(n):
    hist = ecc_hist_cppjit(cw.field(n))
    cp.cuda.get_current_stream().synchronize()
    return cp.asnumpy(hist).astype(np.int64)

Now let's make sure it works. The cold figure is the `cppdef` above plus the first launch, where cppjit compiles the kernel's launcher and the driver turns the PTX into this GPU's machine code, or loads it from its cache.

In [ ]:
warm = cw.measure('cppjit_cuda', run_cppjit, N, compile_s=cppdef_s, cold_note='cppdef and first launch')

### 5. Trace and bytes

One call should be one kernel, one memset and one copy back. The kernel's compulsory traffic is one read of the field, four bytes per voxel. The other seven loads of each thread are served from the caches. The flush at the end of each block is an `atomicAdd` whose result is unused: a reduction executed at the L2 slice that owns the histogram's line, so it never reaches DRAM. The model is that one read. The read over the warm time is the achieved bandwidth against the copy roof.

---

**Quick Docs**

- `cw.copy_roof_gbs()`: the copy bandwidth the intro measured, from `machine.json`.

In [ ]:
cw.trace_call('cppjit_cuda', run_cppjit, N, [ecc_hist_cppjit], header=f'''
import cppjit
BLOCK = {BLOCK}
cppjit.cppdef(cw.ecc_cuda_source(body={KERNEL_BODY!r}))
''')

field_gbs = cw.field(N).nbytes / warm['median_s'] / 1e9
print(f"field read once: {field_gbs:.0f} GB/s = {field_gbs / cw.copy_roof_gbs():.2f} x the copy roof")
cw.roof_report('cppjit_cuda', warm)
cw.save_sweep('cppjit_cuda', run_cppjit)

- One read of the field over the warm time falls below the copy roof. The extra credit asks why.

**EXTRA CREDIT: keep the hardware busy**

§5 showed a gap to the copy roof. [Nsight Compute](https://docs.nvidia.com/nsight-compute/) says why with the kernel's own counters:

- *Waves*: the grid's blocks against the blocks the *SMs* (streaming multiprocessors) hold at once.
- *Speed of light*: the SMs' and the memory system's share of their peaks. The larger share names the limiter.
- *Stall reasons*: what held the warps between instructions.
- *Memory hierarchy*: the load sectors requested against what L1 and L2 serve and what reaches DRAM, the atomics in shared memory, the flush as reductions at L2.

Then let's change the thread mapping. `ecc_hist_rows_kernel` in `cosmic_ecc_variants.cu` is the same body on a 3D grid: `blockIdx.y` and `blockIdx.z` carry $j$ and $i$, so no thread divides to find its voxel, and each thread walks several voxels along $k$. The TODO is its launch on a grid of `(1, n, n)`, one block per row. The block size sets how many voxels a thread walks: sweep it and read the two profiles side by side.

---

**Quick Docs**

- `cw.ncu_report(stage, n, title, run, functions, args, header)`: `ncu --set full` on one launch of the stage's probe, written from `run` when given; prints the metrics that matter here and the top warp stall reasons.

In [ ]:
# EXTRA CREDIT: the kernel of section 3 on the profiler.
cw.ncu_report('cppjit_cuda', N, f'one launch at N = {N}^3:')

- The corner loads request several times the field in sectors. L1 serves the sectors a warp's threads share, L2 most of the rest, and DRAM reads the field once: the traffic the model counts.
- The per-voxel atomics stay in shared memory. More wavefronts than instructions is the price of same-bin collisions inside a warp.
- The flush executes as reductions at L2. Its sectors are a small share of the loads and the DRAM writes stay at the histogram's size.

In [ ]:
# EXTRA CREDIT: the rows kernel.
cppjit.cppdef(cw.ecc_cuda_variants_source())


def ecc_hist_rows(q, block):
    '''Signed histogram of births (int32, length K) on the device, through ecc_hist_rows_kernel.'''
    n = q.shape[0]
    hist = cp.zeros(K, dtype=cp.int32)
    stream = cp.cuda.get_current_stream()
    # TODO: launch cppjit.gbl.ecc_hist_rows_kernel on the grid (1, n, n) with block threads, no dynamic shared memory, on stream, with q, hist and n.
    ...
    return hist


def run_rows(n, block):
    hist = ecc_hist_rows(cw.field(n), block)
    cp.cuda.get_current_stream().synchronize()
    return cp.asnumpy(hist).astype(np.int64)


print(f"{'block':>6} {'voxels/thread':>13} {'warm ms':>9} {'GB/s field':>11} {'max|Δχ|':>8}")
for block in (512, 256, 128, 64):
    r = cw.measure('rows', run_rows, N, block, record=False, quiet=True)
    print(f'{block:>6} {(N + block - 1) // block:>13} {r["median_s"] * 1e3:>9.3f} {cw.field(N).nbytes / r["median_s"] / 1e9:>11.0f} {r["max_abs_diff"]:>8}')

ROWS_BLOCK = 128
cw.ncu_report('rows', N, f'\nthe rows kernel, {(N + ROWS_BLOCK - 1) // ROWS_BLOCK} voxels per thread, one launch at N = {N}^3:',
              run=run_rows, functions=[ecc_hist_rows], args=(ROWS_BLOCK,), header='''
import cppjit
cppjit.cppdef(cw.ecc_cuda_variants_source())
''')

- The SMs' share of peak sits above the memory system's: the kernel is issue-bound, on the divisions and remainders that locate the voxel.
- On the 3D grid the divisions are gone. The memory share leads and the top stall becomes `long_scoreboard`, a wait on a global load: memory-bound, as the bytes model assumed.
- With more voxels per thread the field-read rate climbs, then flattens below the copy roof.

Takeaways:

- The shared-memory histogram turns $8N^3$ global atomics into at most $K$ per block.
- CuPy's arrays and stream went into CUB's function and into the kernel with no copy and no binding.

### 6. Limitations and recap

A name defined in one `cppdef` stays defined for the session, so a second version of a kernel needs a second name. The launch configuration is ours: the grid, the block, the guard and the stream. A field larger than the device is a future direction: the same kernel over slabs with a halo plane, copied from pinned host memory.

Recap:

- CUB's histogram ran from a C++ function compiled in the session, on CuPy's buffers.
- The kernel came from a checked-in source file with its body written in the session.

Next: `95__cosmic_web__synthesis.ipynb` reads the stage files and draws the comparison.